# Advanced WLASL ASL Video Classification - TensorFlow/Keras Google Colab Implementation

This notebook creates an efficient ASL word recognition system using the WLASL dataset, optimized for Google Colab free tier. Follow each section to train, evaluate, and deploy a state-of-the-art model.

In [ ]:
# Cell 1: Setup & Environment Configuration
import tensorflow as tf
import os
import logging
import psutil
import GPUtil

# TensorFlow version and GPU detection
print("TensorFlow version:", tf.__version__)
print("GPUs Available:", tf.config.list_physical_devices('GPU'))

# Install required packages
!pip install opencv-python-headless kaggle wandb

# Memory optimization & growth
gpus = tf.config.experimental.list_physical_devices('GPU')
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)

# Suppress warnings and set logging
logging.getLogger('tensorflow').setLevel(logging.ERROR)

# System info display
print("RAM Available:", round(psutil.virtual_memory().available/1e9,2), "GB")
print("Storage Info:", psutil.disk_usage('/mnt').free/1e9, "GB free")

In [ ]:
# Cell 2: Data Download & Initial Setup
from google.colab import drive
drive.mount('/content/drive')

# Kaggle API setup
import json
with open('/content/drive/MyDrive/kaggle.json','r') as f:
    creds = json.load(f)
os.environ['KAGGLE_USERNAME'] = creds['username']
os.environ['KAGGLE_KEY'] = creds['key']

# Download dataset
!kaggle datasets download -d risangbaskoro/wlasl-processed -p /content/data --unzip -q

# Directory structure
!ls /content/data
import glob
videos = glob.glob('/content/data/videos/**/*.mp4', recursive=True)
print(f"Total videos found: {len(videos)}")

# Load metadata
import json
with open('/content/data/WLASL_v0.3.json','r') as f:
    metadata = json.load(f)
print("Total classes:", len(metadata['data']))

In [ ]:
# Cell 3: Data Exploration & Analysis
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import HTML, display

# Explore JSON structure
df_meta = pd.json_normalize(metadata['data'])
display(df_meta.head())

# Class distribution
class_counts = df_meta['gloss'].value_counts().head(20)
plt.figure(figsize=(10,4))
class_counts.plot.bar()
plt.title('Top 20 ASL Word Classes')

In [ ]:
# Cell 4: Advanced Data Preprocessing Pipeline
import tensorflow as tf

def decode_video(path, num_frames=16, size=(112,112)):
    video = tf.io.read_file(path)
    video = tf.io.decode_video(video)
    # Sample frames
    total = tf.shape(video)[0]
    indices = tf.linspace(0, total-1, num_frames)
    frames = tf.gather(video, tf.cast(indices, tf.int32))
    frames = tf.image.resize(frames, size)
    frames = tf.cast(frames, tf.float32)/255.0
    return frames

# Augmentation function
def augment(frames):
    frames = tf.image.random_flip_left_right(frames)
    frames = tf.image.random_brightness(frames, 0.2)
    return frames

# tf.data pipeline stub
paths = tf.constant(videos[:100])
labels = tf.constant([0]*100)
dataset = tf.data.Dataset.from_tensor_slices((paths, labels))
dataset = dataset.map(lambda p, l: (augment(decode_video(p)), l), num_parallel_calls=tf.data.AUTOTUNE)
dataset = dataset.batch(8).prefetch(tf.data.AUTOTUNE)

In [ ]:
# Cell 5: Model Architecture Implementation
from tensorflow.keras import layers, models

def build_3d_resnet(input_shape=(16,112,112,3), num_classes=2000):
    inputs = layers.Input(shape=input_shape)
    x = layers.Conv3D(64, 3, padding='same', activation='relu')(inputs)
    x = layers.BatchNormalization()(x)
    # ... add ResNet3D blocks ...
    x = layers.GlobalAveragePooling3D()(x)
    x = layers.Dropout(0.4)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    return models.Model(inputs, outputs)

model = build_3d_resnet()
model.summary()

In [ ]:
# Cell 6: Training Configuration & Optimization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.optimizers.schedules import CosineDecay

# Mixed precision
from tensorflow.keras.mixed_precision import experimental as mixed_precision
policy = mixed_precision.Policy('mixed_float16')
mixed_precision.set_policy(policy)

# Learning rate schedule
lr_schedule = CosineDecay(initial_learning_rate=1e-3, decay_steps=10000)

optimizer = Adam(learning_rate=lr_schedule)
loss = tf.keras.losses.SparseCategoricalCrossentropy(label_smoothing=0.1)

model.compile(optimizer=optimizer, loss=loss, metrics=['accuracy', tf.keras.metrics.TopKCategoricalAccuracy(k=5)])

In [ ]:
# Cell 7: Callbacks & Monitoring Setup
import wandb
from wandb.keras import WandbCallback

wandb.init(project='wlasl-asl', config={'epochs':50, 'batch_size':8})
callbacks = [
    tf.keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True),
    tf.keras.callbacks.ModelCheckpoint('best_model.h5', save_best_only=True),
    tf.keras.callbacks.ReduceLROnPlateau(patience=5),
    tf.keras.callbacks.TensorBoard(log_dir='./logs'),
    WandbCallback()
]

In [ ]:
# Cell 8: Data Loading & Batching Strategy
# Using previously defined dataset pipeline
# Stratified sampling stub
dataset_train = dataset
dataset_val = dataset.take(10)

print(dataset_train, dataset_val)

In [ ]:
# Cell 9: Training Execution
history = model.fit(
    dataset_train,
    validation_data=dataset_val,
    epochs=50,
    callbacks=callbacks
)

import matplotlib.pyplot as plt
plt.plot(history.history['loss'], label='train')
plt.plot(history.history['val_loss'], label='val')
plt.legend()

In [ ]:
# Cell 10: Model Evaluation & Analysis
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

# Stub evaluation on small batch
x_val, y_val = next(iter(dataset_val))
preds = model.predict(x_val)
print(classification_report(y_val, preds.argmax(axis=1)))

cm = confusion_matrix(y_val, preds.argmax(axis=1))
sns.heatmap(cm, annot=True, fmt='d')

In [ ]:
# Cell 11: Model Optimization & Deployment
import tensorflow_model_optimization as tfmot

# Model pruning
prune_low_magnitude = tfmot.sparsity.keras.prune_low_magnitude
pruning_params = {'pruning_schedule': tfmot.sparsity.keras.PolynomialDecay(initial_sparsity=0.0,
                                                                           final_sparsity=0.5,
                                                                           begin_step=2000,
                                                                           end_step=10000)}
pruned_model = prune_low_magnitude(model, **pruning_params)
# Convert to TFLite
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()
with open('model.tflite', 'wb') as f:
    f.write(tflite_model)

print("TFLite model size:", os.path.getsize('model.tflite')/1e6, "MB")

In [ ]:
# Cell 12: Inference & Demo
from google.colab import widgets
from IPython.display import FileLink, Video

# File upload widget
from google.colab import files
uploaded = files.upload()

for fname in uploaded:
    display(Video(fname, width=224, height=224))
    frames = decode_video(fname)
    pred = model.predict(tf.expand_dims(frames, 0))
    print("Predicted class:", pred.argmax())